### ✅ **QUESTÃO 8.1 — CÓDIGO COMPLETO**

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# =========================
# 1. Carregar dados
# =========================
vendas = pd.read_csv('../data/raw/vendas_2023_2024.csv')
produtos = pd.read_csv('../data/processed/produtos_clean.csv')

# =========================
# 2. Padronizar colunas
# =========================
vendas = vendas.rename(columns={'id_product': 'product_id'})
produtos = produtos.rename(columns={'code': 'product_id'})

# Garantir mesmo tipo
vendas['product_id'] = vendas['product_id'].astype(str).str.strip()
produtos['product_id'] = produtos['product_id'].astype(str).str.strip()

# =========================
# 3. Merge
# =========================
df = vendas.merge(produtos, on='product_id', how='left')

# =========================
# 4. Criar matriz usuário-item (binária)
# =========================
df['comprou'] = 1

matriz = (
    df.groupby(['id_client', 'product_id'])['comprou']
    .max()
    .unstack(fill_value=0)
)

# =========================
# 5. Similaridade (produto x produto)
# =========================
matriz_produto = matriz.T  # produto x cliente

similaridade = cosine_similarity(matriz_produto)

similaridade_df = pd.DataFrame(
    similaridade,
    index=matriz_produto.index,
    columns=matriz_produto.index
)

# =========================
# 6. Mapear nomes dos produtos
# =========================
mapa_produtos = produtos.set_index('product_id')['name']

# =========================
# 7. Escolher produto GPS de referência
# =========================
gps_produtos = mapa_produtos[mapa_produtos.str.startswith("GPS", na=False)]

# pegar o mais frequente (mais vendido)
gps_mais_vendido = df[df['product_id'].isin(gps_produtos.index)]['product_id'].value_counts().idxmax()

print(f"Produto GPS escolhido: {mapa_produtos[gps_mais_vendido]}")

# =========================
# 8. Ranking de similaridade
# =========================
ranking = (
    similaridade_df[gps_mais_vendido]
    .sort_values(ascending=False)
    .drop(gps_mais_vendido)  # remover ele mesmo
)

top5 = ranking.head(5)

# =========================
# 9. Mostrar resultado
# =========================
resultado = pd.DataFrame({
    'product_id': top5.index,
    'similaridade': top5.values,
    'nome_produto': top5.index.map(mapa_produtos)
})

print("\nTop 5 produtos similares:")
print(resultado)

Produto GPS escolhido: GPS Furuno Magnum Drift

Top 5 produtos similares:
  product_id  similaridade                                   nome_produto
0        137      0.880078                        Âncora Danforth Impulse
1         26      0.861381       Piloto Automático Furuno Core Boost Flux
2        138      0.846432    Boia de Arqueamento Delta Peak Boost Thrust
3         51      0.841110           Motor Diesel Tohatsu Evo Zenith 16HP
4         98      0.836046  Motor Diesel Parsun Velocity Abyss Orca 235HP


### 🎯 **QUESTÃO 8.2 — VALIDAÇÃO**

### 🧠 **QUESTÃO 8.3 — EXPLICAÇÃO**

Como o produto “GPS Garmin Striker 4” não estava presente na base, foi utilizado o produto GPS mais representativo (mais vendido), garantindo consistência estatística na análise.

📌 **1. Como a matriz foi construída**

A matriz usuário–item foi construída utilizando:
  * Linhas representando os clientes (id_client)
  * Colunas representando os produtos (id_product)
  * Valores binários:
    * 1 para indicar que o cliente comprou o produto ao menos uma vez
    * 0 caso contrário

A quantidade comprada foi desconsiderada, focando apenas na presença ou ausência de compra.

📌 **2. O que significa a similaridade de cosseno**

A similaridade de cosseno mede o grau de semelhança entre dois produtos com base no comportamento de compra dos clientes.

Nesse contexto:
  * Produtos com alta similaridade são frequentemente comprados pelos mesmos clientes
  * Isso indica uma forte relação de consumo entre os itens

📌 **3. Limitação do método**

Uma limitação desse método é que ele não considera:
  * Sequência de compra
  * Contexto de uso
  * Quantidade comprada

Além disso, produtos com poucas interações podem gerar similaridades menos confiáveis.